In [ ]:
# ! pip install pybaseball

# Prepare Environment

In [ ]:
# Check for local CSV; if missing, download from Firebase using pyasebase/pyrebase
import os
import pandas as pd
from pybaseball import pitching_stats
import numpy as np

csv_path = "./data/pitching_stats.csv"

if os.path.exists(csv_path):
    print(f"Found {csv_path}, reading into DataFrame.")
    data = pd.read_csv(csv_path)
else:
    print(f"{csv_path} not found — attempting to download using pybaseball.")
    qual = 10
    start_season = 2015
    end_season = 2025
    data = pitching_stats(start_season=start_season, end_season=end_season, qual=qual)
    # data.to_csv('./data/pitching_stats.csv', index=False)

In [ ]:
from src.cnn_pitcher_model import CNNPitchingDataset, CNNPitchingPredictionDataset, CNNPitchingModel

# Prepare Training Data

In [ ]:
# Process data for 3D tensor - organized by player-season combination
import torch
import numpy as np
import pandas as pd

# Select relevant columns
features = [
    'Age',
    'ERA',
    'G',
    'GS',
    'IP',
    'TBF',
    'HR',
    'BB',
    'IBB',
    'HBP',
    'SO',
    'GB',
    'GB%',
    'FB',
    'FB%',
    'LD',
    'LD%',
    'IFFB',
    'Pitches',
    'BABIP',
    'WHIP',
    'FIP',
    'xFIP',
    'SIERA',
    'CStr%',
    'CSW%',
    'Barrels',
    'Barrel%',
    'HardHit',
    'HardHit%',
]

target = 'WAR'
nlookbacks = 5
# Qualification parameter: minimum IP in target season to include in training set
min_qual_ip = 50  # Set to > 0 to filter by innings pitched (e.g., 50 for minimum 50 IP)

cnn_pitching_data = CNNPitchingDataset(
    data = data,
    features=features,
    target = target,
    nlookbacks = nlookbacks,
)


X = torch.tensor(cnn_pitching_data.sequences, dtype=torch.float32)
y = torch.tensor(cnn_pitching_data.target, dtype=torch.float32)

# Create metadata DataFrame indexed by player_id and target_season
metadata_df = pd.DataFrame(cnn_pitching_data.metadata)
metadata_df = metadata_df.set_index(['player_id', 'target_season'])

print(f"Tensor shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Metadata rows: {len(metadata_df)}")
print(f"Qualification filter: min_qual_ip = {min_qual_ip}")
print(f"\nFirst few metadata entries:")
print(metadata_df.head())
print(f"\nSample sequence for entry 0:\n{X[0]}")
print(f"Corresponding target: {y[0]}")


# Prepare Prediction Data

In [ ]:
# Create prediction set for 2025 season
pred_season = 2025

cnn_prediction_datatset = CNNPitchingPredictionDataset(
    data = data,
    pred_season = pred_season, 
    features = features,
)
# Convert to torch tensors
X_pred = torch.tensor(np.array(cnn_prediction_datatset.sequences), dtype=torch.float32)
pred_metadata_df = cnn_prediction_datatset.metadata

print(f"Prediction set size: {X_pred.shape}")
print(f"Number of players with {pred_season} data: {len(pred_metadata_df)}")
print(f"\nFirst few prediction metadata entries:")
print(pred_metadata_df.head())


# Define Model using 1D-CNN

In [ ]:
# 1D-CNN Model for WAR Prediction
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

def prepare_data(X, y, test_size=0.2, batch_size=32):
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)

    # Normalize features
    scaler = StandardScaler()
    X_train_reshaped = X_train.view(X_train.shape[0], -1).numpy()
    X_test_reshaped = X_test.view(X_test.shape[0], -1).numpy()

    X_train_scaled = scaler.fit_transform(X_train_reshaped)
    X_test_scaled = scaler.transform(X_test_reshaped)

    X_train = torch.tensor(X_train_scaled.reshape(X_train.shape), dtype=torch.float32)
    X_test = torch.tensor(X_test_scaled.reshape(X_test.shape), dtype=torch.float32)

    # Create datasets
    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader, scaler

class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

def train_model(model, train_loader, test_loader, num_epochs=100, learning_rate=0.001, device='cpu', patience=15, early_stopping_enabled=True):
    model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    early_stopping = EarlyStopping(patience=patience) if early_stopping_enabled else None

    train_losses = []
    test_losses = []
    best_model_state = None
    best_test_loss = float('inf')

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs.squeeze(), targets)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)

        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)

        # Validation
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for inputs, targets in test_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs.squeeze(), targets)
                test_loss += loss.item() * inputs.size(0)

        test_loss /= len(test_loader.dataset)
        test_losses.append(test_loss)

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}')

        # Track best model
        if test_loss < best_test_loss:
            best_test_loss = test_loss
            best_model_state = model.state_dict().copy()

        # Early stopping check
        if early_stopping_enabled:
            early_stopping(test_loss)
            if early_stopping.early_stop:
                print(f'Early stopping at epoch {epoch+1}. Best Test Loss: {best_test_loss:.4f}')
                break

    # Load best model if early stopping was used
    if best_model_state is not None and early_stopping_enabled:
        model.load_state_dict(best_model_state)

    return train_losses, test_losses

def plot_losses(train_losses, test_losses):
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(test_losses, label='Test Loss')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.show()

# Example usage:
# model = PitchingCNN(input_channels=6, seq_length=3)
# train_loader, test_loader, scaler = prepare_data(X, y)
# train_losses, test_losses = train_model(model, train_loader, test_loader, device='cpu')
# plot_losses(train_losses, test_losses)

# Train Model

In [ ]:
model = CNNPitchingModel(input_channels=len(features), seq_length=3)
print(model)

In [ ]:
# Train the 1D-CNN model
# Check if MPS is available
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

# Create model
model = CNNPitchingModel(input_channels=len(features), seq_length=nlookbacks)

# Prepare data and save scaler
train_loader, test_loader, scaler = prepare_data(X, y, batch_size=256)

# Train model
train_losses, test_losses = train_model(model, train_loader, test_loader, learning_rate=1e-3, num_epochs=100, device=device, patience=20, early_stopping_enabled=True)

# Plot losses
plot_losses(train_losses, test_losses)

# Make predictions on test set
model.eval()
predictions = []
actuals = []
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        predictions.extend(outputs.cpu().numpy().flatten())
        actuals.extend(targets.numpy())

# Calculate metrics
from sklearn.metrics import mean_squared_error, r2_score
mse = mean_squared_error(actuals, predictions)
r2 = r2_score(actuals, predictions)
print(f"Test MSE: {mse:.4f}")
print(f"Test R²: {r2:.4f}")

# Plot predictions vs actuals
plt.figure(figsize=(8, 6))
plt.scatter(actuals, predictions, alpha=0.5)
plt.plot([min(actuals), max(actuals)], [min(actuals), max(actuals)], 'r--')
plt.xlabel('Actual WAR')
plt.ylabel('Predicted WAR')
plt.title('Predicted vs Actual WAR')
plt.show()

# Store scaler and model for later use
print("Scaler and model saved for predictions")

# Generate Predictions for 2026

In [ ]:
# Store predictions using saved scaler

# Scale prediction data using the fitted scaler
X_pred_reshaped = X_pred.view(X_pred.shape[0], -1).numpy()
X_pred_scaled = scaler.transform(X_pred_reshaped)
X_pred_tensor = torch.tensor(X_pred_scaled.reshape(X_pred.shape), dtype=torch.float32)

# Make predictions
model.eval()
with torch.no_grad():
    predictions = model(X_pred_tensor.to(device))

# Convert to numpy
pred_war = predictions.cpu().numpy().flatten()

# Add to results
results_df = pred_metadata_df.copy()
results_df['predicted_war_2026'] = pred_war

print("Predictions (now in correct WAR scale):")
print(results_df.head())
print(f"\nPredicted WAR range: [{pred_war.min():.2f}, {pred_war.max():.2f}]")

# Save predictions to CSV
results_df.to_csv('./results/cnn/2026_war_predictions.csv', index=True)
print(f"\nPredictions saved to ./results/cnn/2026_war_predictions.csv")

In [ ]:
pd.set_option('display.max_rows', 100)
csv_path = "./results/cnn/2026_war_predictions.csv"
if os.path.exists(csv_path):
    print(f"Found {csv_path}, reading into DataFrame.")
    results_df = pd.read_csv(csv_path)

data_2025 = data[data['Season'] == 2025]
results_with_name = results_df.merge(data_2025[['IDfg','Name',]], left_on=['player_id'], right_on=['IDfg'], how='left')

In [ ]:
results_with_name.sort_values('predicted_war_2026', ascending=False).head(100)[['Name', 'IDfg', 'predicted_war_2026',]]

---